# Feature Predictivity Test

Test whether features have predictive power for targets using real market data.

**Configuration:**
- Symbols: QQQ, SPY, IWM, DIA, TLT, GLD
- Training: SLIDING window (N samples)
- Test: Next 2 periods

**Pass criteria:** R² > 0

**Output:** CSV results saved to `data_work/feature_predictivity_results.csv`

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from datetime import datetime

from core.search_data import search_data
from norm.norm_utils import load_normalized_df
from features.base_dataframe import BaseDataFrame
from features.features_utils import FeatureType
from fitting.fitting_models import TimeSeriesModelTrainer, TrainingConfig
from fitting.fitting_core import TrainingSplitType, TaskType, RetrainMode
from fitting.models import ModelFactory, ModelType
from fitting.fitting_metrics import RegressionMetric
from core.enums import g_close_col, g_open_col, g_index_col

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

MINUTES_PER_DAY = 1440
HOURS_PER_DAY = 24

## 1. Configuration

In [2]:
# Symbols to test
SYMBOLS = ['QQQ', 'SPY', 'IWM', 'DIA', 'TLT', 'GLD']

# Training configuration
SLIDING_WINDOW_SIZE = 256  # Number of samples in training window
TEST_PERIODS = 2  # Number of periods to test (retrain_period)

# Targets to test
TARGETS = ['T_overnight_perf', 'T_close_to_close']

# Models to test
MODELS_TO_TEST = [ModelType.XGB, ModelType.RF_SK, ModelType.DT_SK]

# Feature periods for technical indicators
RSI_PERIODS = [1,2,4,8,16,32,64,128,256]
EMA_PERIODS = [1,2,4,8,16,32,64,128,256]
VOL_PERIODS = [1,2,4,8,16,32,64,128,256]
ROC_PERIODS = [1,2,4,8,16,32,64,128,256]

print(f"Configuration:")
print(f"  Symbols: {SYMBOLS}")
print(f"  Sliding window size: {SLIDING_WINDOW_SIZE}")
print(f"  Test periods: {TEST_PERIODS}")
print(f"  Targets: {TARGETS}")
print(f"  Models: {[m.name for m in MODELS_TO_TEST]}")

Configuration:
  Symbols: ['QQQ', 'SPY', 'IWM', 'DIA', 'TLT', 'GLD']
  Sliding window size: 256
  Test periods: 2
  Targets: ['T_overnight_perf', 'T_close_to_close']
  Models: ['XGB', 'RF_SK', 'DT_SK']


## 2. Data Loading Functions

In [3]:
def load_symbol_data(symbol: str, freq: str = "candle_1hour") -> pd.DataFrame:
    """Load hourly candle data for a symbol."""
    files = search_data(p_symbol=symbol, p_data_freq=freq)
    if not files:
        raise ValueError(f"No data found for symbol {symbol}")
    df = load_normalized_df(str(files[0].path))
    print(f"  Loaded {symbol}: {len(df)} rows")
    return df


def load_all_symbols(symbols: list) -> dict:
    """Load data for all symbols."""
    data = {}
    print("Loading data for symbols:")
    for symbol in symbols:
        try:
            data[symbol] = load_symbol_data(symbol)
        except Exception as e:
            print(f"  ERROR loading {symbol}: {e}")
    return data

## 3. Feature Engineering Functions

In [4]:
def generate_all_features(df: pd.DataFrame) -> tuple:
    """
    Generate ALL available features for the dataframe.
    
    Returns:
        tuple: (df, feature_cols, day_feature_cols)
    """
    bdf = BaseDataFrame(p_df=df, p_verbose=False)
    
    # Technical indicators with multiple periods
    bdf.add_feature(FeatureType.RSI, periods=RSI_PERIODS)
    bdf.add_feature(FeatureType.SPREAD_REL_EMA, periods=EMA_PERIODS)
    bdf.add_feature(FeatureType.DIFF_REL_EMA_MID, periods=EMA_PERIODS)
    bdf.add_feature(FeatureType.HIST_VOLATILITY, periods=VOL_PERIODS)
    bdf.add_feature(FeatureType.ROC, periods=ROC_PERIODS)
    bdf.add_feature(FeatureType.ADI)
    
    # Volume features
    bdf.add_feature(FeatureType.VOLUME)
    bdf.add_feature(FeatureType.V_OBV)
    bdf.add_feature(FeatureType.V_MA_RATIO, periods=[15, 60])
    
    # Lagged deltas
    bdf.add_feature(FeatureType.LAG_DELTAS,n_lags = 48,n_minutes = 1)
    
    df = bdf.get_dataframe()
    feature_cols = bdf.get_feature_columns()
    
    # Add day of week
    base = pd.Timestamp("2000-01-01")
    df["datetime"] = base + pd.to_timedelta(df[g_index_col], unit="m")
    df["F_day_of_week"] = df["datetime"].dt.dayofweek.astype(np.float32)
    
    # Add hour of day
    df["F_hour_of_day"] = df["datetime"].dt.hour.astype(np.float32)

    day_feature_cols = ["F_day_of_week", "F_hour_of_day"]
    
    feature_cols = feature_cols 
    
    print(f"  Generated {len(feature_cols)} features total")
    return df, feature_cols, day_feature_cols

## 4. Target Definition

In [5]:
def compute_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Compute target columns for the dataframe."""
    # Overnight performance: (next_open - close) / close
    df["T_overnight_perf"] = (df[g_open_col].shift(-1) - df[g_close_col]) / df[g_close_col]
    
    # Close to close: (next_close - close) / close
    df["T_close_to_close"] = (df[g_close_col].shift(-1) - df[g_close_col]) / df[g_close_col]
    
    # Replace inf values with nan
    df["T_overnight_perf"] = df["T_overnight_perf"].replace([np.inf, -np.inf], np.nan)
    df["T_close_to_close"] = df["T_close_to_close"].replace([np.inf, -np.inf], np.nan)
    
    return df

## 5. Data Preparation

In [6]:
def prepare_data(df: pd.DataFrame, feature_cols: list, target_col: str, 
                 eod_only: bool = True) -> tuple:
    """
    Prepare feature matrix X and target vector y.
    
    Args:
        df: DataFrame with features and targets
        feature_cols: List of feature column names
        target_col: Target column name
        eod_only: If True, only use end-of-day candles
    
    Returns:
        tuple: (X, y, valid_mask)
    """
    # Add EOD detection if needed
    if eod_only and "is_eod" not in df.columns:
        df["day_num"] = df[g_index_col] // MINUTES_PER_DAY
        df["next_day_num"] = df["day_num"].shift(-1)
        df["is_eod"] = df["day_num"] != df["next_day_num"]
        df.loc[df["is_eod"].isna(), "is_eod"] = False
    
    # Create valid mask
    valid_mask = ~df[feature_cols].isna().any(axis=1)
    valid_mask &= ~df[target_col].isna()
    
    if eod_only:
        valid_mask &= df["is_eod"]
    
    X = df.loc[valid_mask, feature_cols].values.astype(np.float32)
    y = df.loc[valid_mask, target_col].values.astype(np.float32)
    
    return X, y, valid_mask

## 6. Model Training Function

In [7]:
def fit_model(X, y, model_type: ModelType, train_ratio: float = 0.2,
              sliding_window_size: int = 126, retrain_period: int = 2) -> tuple:
    """
    Train a model with SLIDING window periodic retraining.
    
    Args:
        X: Feature matrix
        y: Target vector
        model_type: Type of model to train
        train_ratio: Fraction of data for initial training
        sliding_window_size: Size of training window
        retrain_period: Number of periods between retraining
    
    Returns:
        tuple: (trainer, metrics)
    """
    config = TrainingConfig(
        mode=TrainingSplitType.PERIODIC_RETRAIN,
        train_ratio=train_ratio,
        retrain_period=retrain_period,
        retrain_mode=RetrainMode.SLIDING,
        sliding_window_size=sliding_window_size,
        normalization="standardize",
    )
    
    model = ModelFactory.create_model(
        model_type=model_type,
        task_type=TaskType.REGRESSION,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
    )
    
    trainer = TimeSeriesModelTrainer(
        model=model,
        config=config,
        task_type=TaskType.REGRESSION,
        verbose=False,
    )
    
    metrics = trainer.fit(X, y)
    return trainer, metrics

## 7. Single Symbol Test

In [8]:
def run_single_test(symbol: str, target_col: str, model_type: ModelType,
                    sliding_window_size: int = 126, 
                    retrain_period: int = 2,
                    eod_only: bool = True) -> dict:
    """
    Run a single predictivity test for a symbol, target, and model.
    
    Returns:
        dict: Results including R², correlation, MSE, etc.
    """
    print(f"\n{'='*60}")
    print(f"Testing {symbol} | {target_col} | {model_type.name}")
    print(f"{'='*60}")
    
    # Load data
    df = load_symbol_data(symbol)
    
    # Generate features
    df, feature_cols, day_feature_cols = generate_all_features(df)
    
    # Compute targets
    df = compute_targets(df)
    
    # Prepare data
    X, y, valid_mask = prepare_data(df, feature_cols, target_col, eod_only=eod_only)
    print(f"  Prepared {X.shape[0]} samples with {X.shape[1]} features")
    
    if X.shape[0] < 100:
        print(f"  WARNING: Too few samples ({X.shape[0]})")
        return None
    
    # Train model
    train_ratio = max(0.1, sliding_window_size / X.shape[0])
    trainer, metrics = fit_model(
        X, y, model_type,
        train_ratio=train_ratio,
        sliding_window_size=sliding_window_size,
        retrain_period=retrain_period
    )
    
    # Calculate additional metrics
    predictions = trainer.predict(X)
    correlation = float(np.corrcoef(y, predictions)[0, 1])
    
    # Get chunk-level R²s for analysis
    chunk_r2s = []
    if metrics.retrain_history:
        for entry in metrics.retrain_history:
            chunk_metrics = entry.get('test_chunk_metrics', {})
            if chunk_metrics:
                chunk_r2s.append(chunk_metrics.get('r2', 0))
    
    result = {
        'symbol': symbol,
        'target': target_col,
        'model': model_type.name,
        'n_samples': int(X.shape[0]),
        'n_features': int(X.shape[1]),
        'train_r2': float(metrics.train_metrics.get('r2', 0)),
        'train_score': float(metrics.train_score),
        'test_r2': float(metrics.test_metrics.get('r2', 0) if metrics.test_metrics else 0),
        'test_score': float(metrics.test_score),
        'correlation': correlation,
        'mse': float(metrics.test_metrics.get('mse', 0) if metrics.test_metrics else 0),
        'rmse': float(metrics.test_metrics.get('rmse', 0) if metrics.test_metrics else 0),
        'mae': float(metrics.test_metrics.get('mae', 0) if metrics.test_metrics else 0),
        'chunk_r2_mean': float(np.mean(chunk_r2s)) if chunk_r2s else 0,
        'chunk_r2_std': float(np.std(chunk_r2s)) if chunk_r2s else 0,
        'n_chunks': len(chunk_r2s),
    }
    
    # Print summary
    print(f"  Train R²: {result['train_r2']:.4f}")
    print(f"  Test R²:  {result['test_r2']:.4f}")
    print(f"  Correlation: {result['correlation']:.4f}")
    print(f"  Pass (R²>0): {'YES' if result['test_r2'] > 0 else 'NO'}")
    
    return result

## 8. Run All Tests

In [9]:
# Collect all results
all_results = []

SYMBOLS = ['QQQ']

for symbol in SYMBOLS:
    for target in TARGETS:
        for model_type in MODELS_TO_TEST:
            try:
                result = run_single_test(
                    symbol=symbol,
                    target_col=target,
                    model_type=model_type,
                    sliding_window_size=SLIDING_WINDOW_SIZE,
                    retrain_period=TEST_PERIODS,
                    eod_only=True
                )
                if result:
                    all_results.append(result)
            except Exception as e:
                print(f"  ERROR: {e}")
                continue


Testing QQQ | T_overnight_perf | XGB
  Loaded QQQ: 92274 rows
  Generated 100 features total
  Prepared 6550 samples with 100 features
  Train R²: -0.1443
  Test R²:  -0.3245
  Correlation: 0.0494
  Pass (R²>0): NO

Testing QQQ | T_overnight_perf | RF_SK
  Loaded QQQ: 92274 rows
  Generated 100 features total
  Prepared 6550 samples with 100 features
  ERROR: RandomForestRegressor.__init__() got an unexpected keyword argument 'learning_rate'

Testing QQQ | T_overnight_perf | DT_SK
  Loaded QQQ: 92274 rows
  Generated 100 features total
  Prepared 6550 samples with 100 features
  ERROR: DecisionTreeRegressor.__init__() got an unexpected keyword argument 'n_estimators'

Testing QQQ | T_close_to_close | XGB
  Loaded QQQ: 92274 rows
  Generated 100 features total
  Prepared 6550 samples with 100 features
  Train R²: -0.2341
  Test R²:  -0.3001
  Correlation: 0.0496
  Pass (R²>0): NO

Testing QQQ | T_close_to_close | RF_SK
  Loaded QQQ: 92274 rows
  Generated 100 features total
  Prepared 

## 9. Results Summary

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(all_results)

# Sort by test R²
results_df = results_df.sort_values('test_r2', ascending=False)

# Display summary
print("\n" + "="*80)
print("FEATURE PREDICTIVITY TEST RESULTS")
print("="*80)
print(f"\nTotal tests: {len(results_df)}")
print(f"Tests passing (R² > 0): {(results_df['test_r2'] > 0).sum()}")
print(f"Tests failing (R² <= 0): {(results_df['test_r2'] <= 0).sum()}")

# Summary by symbol
print("\n--- Summary by Symbol ---")
symbol_summary = results_df.groupby('symbol').agg({
    'test_r2': ['mean', 'max'],
    'correlation': ['mean', 'max']
}).round(4)
symbol_summary.columns = ['avg_r2', 'max_r2', 'avg_corr', 'max_corr']
print(symbol_summary)

# Summary by target
print("\n--- Summary by Target ---")
target_summary = results_df.groupby('target').agg({
    'test_r2': ['mean', 'max'],
    'correlation': ['mean', 'max']
}).round(4)
target_summary.columns = ['avg_r2', 'max_r2', 'avg_corr', 'max_corr']
print(target_summary)

# Summary by model
print("\n--- Summary by Model ---")
model_summary = results_df.groupby('model').agg({
    'test_r2': ['mean', 'max'],
    'correlation': ['mean', 'max']
}).round(4)
model_summary.columns = ['avg_r2', 'max_r2', 'avg_corr', 'max_corr']
print(model_summary)

## 10. Detailed Results Table

In [ ]:
# Display all results sorted by test R²
display_cols = ['symbol', 'target', 'model', 'n_samples', 'train_r2', 'test_r2', 
                'correlation', 'mse', 'pass']
results_df['pass'] = results_df['test_r2'] > 0

# Format for display
display_df = results_df[display_cols].copy()
for col in ['train_r2', 'test_r2', 'correlation', 'mse']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

pd.set_option('display.max_rows', 100)
print(display_df.to_string(index=False))

## 11. Save Results to CSV

In [ ]:
# Generate filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = Path("data_work") / f"feature_predictivity_results_{timestamp}.csv"

# Ensure directory exists
output_file.parent.mkdir(parents=True, exist_ok=True)

# Save results
results_df.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")

# Also save the latest version without timestamp
latest_file = Path("data_work") / "feature_predictivity_results.csv"
results_df.to_csv(latest_file, index=False)
print(f"Latest results also saved to: {latest_file}")

## 12. Visualizations

In [ ]:
import matplotlib.pyplot as plt

# Filter to XGB results for consistent comparison
xgb_results = results_df[results_df['model'] == 'XGB'].copy()

# Plot: Test R² by Symbol and Target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Test R² by Symbol
symbol_r2 = xgb_results.groupby('symbol')['test_r2'].mean().sort_values()
axes[0].barh(symbol_r2.index, symbol_r2.values)
axes[0].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Average Test R²')
axes[0].set_title('XGBoost: Average Test R² by Symbol')
for i, v in enumerate(symbol_r2.values):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center')

# Subplot 2: Test R² by Target
target_r2 = xgb_results.groupby('target')['test_r2'].mean().sort_values()
axes[1].barh(target_r2.index, target_r2.values)
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Average Test R²')
axes[1].set_title('XGBoost: Average Test R² by Target')
for i, v in enumerate(target_r2.values):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Test R² by Model
fig, ax = plt.subplots(figsize=(10, 5))

model_r2 = results_df.groupby('model')['test_r2'].mean().sort_values()
colors = ['green' if v > 0 else 'red' for v in model_r2.values]
ax.barh(model_r2.index, model_r2.values, color=colors)
ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Average Test R²')
ax.set_title('Average Test R² by Model (Green = Pass, Red = Fail)')

for i, v in enumerate(model_r2.values):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

## 13. Conclusions

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)

# Best performing combinations
print("\n--- Best Performing (Top 5 by Test R²) ---")
top5 = results_df.nlargest(5, 'test_r2')[['symbol', 'target', 'model', 'test_r2', 'correlation']]
for idx, row in top5.iterrows():
    print(f"  {row['symbol']:5s} | {row['target']:20s} | {row['model']:10s} | R²={row['test_r2']:.4f} | Corr={row['correlation']:.4f}")

# Worst performing combinations
print("\n--- Worst Performing (Bottom 5 by Test R²) ---")
bottom5 = results_df.nsmallest(5, 'test_r2')[['symbol', 'target', 'model', 'test_r2', 'correlation']]
for idx, row in bottom5.iterrows():
    print(f"  {row['symbol']:5s} | {row['target']:20s} | {row['model']:10s} | R²={row['test_r2']:.4f} | Corr={row['correlation']:.4f}")

# Overall statistics
print("\n--- Overall Statistics ---")
print(f"  Total tests: {len(results_df)}")
print(f"  Passing tests (R² > 0): {(results_df['test_r2'] > 0).sum()} ({(results_df['test_r2'] > 0).mean()*100:.1f}%)")
print(f"  Failing tests (R² <= 0): {(results_df['test_r2'] <= 0).sum()} ({(results_df['test_r2'] <= 0).mean()*100:.1f}%)")
print(f"  Average Test R²: {results_df['test_r2'].mean():.4f}")
print(f"  Max Test R²: {results_df['test_r2'].max():.4f}")
print(f"  Average Correlation: {results_df['correlation'].mean():.4f}")